# Story 2 View 2: who produced the benchmark scores behind the capability index?

**Question:** For the models in Story 2 View 1, how many benchmark scores were run by Epoch AI, how many come from a third party and how many come from a paper or model card published by a developer, by release year and by access group?

**Method in one paragraph.** Download the Epoch Benchmarking Hub ZIP. Stack the benchmark files listed in the Hub metadata (58 in the file used for the page) into one table of scores. Label each score by who produced it. Match the Hub model names to the models of the capability file. Keep one score per model, benchmark and kind of source (the highest, if there are several). Count scores by release year and access group (the same `access_group` rule as Story 2 View 1). Write one CSV that the app reads.

**Data licence:** Epoch AI Benchmarking Hub, Creative Commons Attribution 4.0. Credit: Epoch AI, 'Capabilities & benchmarking', epoch.ai/benchmarks. Benchmark questions and answers belong to their creators, and data from external projects keeps its own licence.

**Counting rule (fixed):** One score = one model, one benchmark and one kind of source. Versions of a model (for example high, low and max effort) are merged and the highest score is kept. A benchmark with two kinds of source for the same model counts twice. A benchmark with no result is not counted.

**Review status:** the list of source labels (`manual/source_evidence_labels.csv`) and the model name matches (`processed/hub_name_matches.csv`) were read by one reader only. The `second_reviewer` column of the match file is empty.

## At a glance: links, authorisation, download, timing

| Item | Detail |
|---|---|
| Kind | Download of one ZIP file (the Epoch Benchmarking Hub data), plus one hand-made table in `manual/` |
| Where it comes from | Epoch AI, Benchmarking Hub. https://epoch.ai/data/benchmark_data.zip |
| Licence and authorisation | Creative Commons Attribution 4.0 for the Hub data. No account or key needed. Details in Section 1 |
| Download size | about 2.3 MB |
| Needs first | `../eci_scores/processed/model_capability.csv`. Run the eci_scores notebook of this story before this one. If that file is missing, the notebook uses `../../data/model_capability.csv` |
| Expected time | The download is one request. The rest of the notebook works on small tables in memory. The last cells print the measured time |
| Data as of | `processed/hub_data_currency.csv` and `raw/retrieval_log.json` |
| To update | Set `REFRESH = True` in Section 3 and run all cells. Read "How to update" at the end |


## 1. Source, reference links and authorisation

**Source.** Epoch AI, Benchmarking Hub. Data file: https://epoch.ai/data/benchmark_data.zip. Page: https://epoch.ai/benchmarks.

**Licence.** Creative Commons Attribution 4.0 for the Hub data. Credit: Epoch AI, 'Capabilities & benchmarking', epoch.ai/benchmarks. Benchmark questions and answers belong to their creators, and data from external projects keeps its own licence.

**Authorisation.** The file is public. No account, key or login is needed. The notebook sends a request that names the project (`User-Agent`) and stores the file exactly as received in `raw/`.


## 2. Table structure

### 2.1 Structure of the source file `benchmark_data.zip`

A ZIP archive with `benchmark_metadata.csv` and one CSV file for each benchmark. The notebook uses these columns:

| Column | Where | Meaning |
|---|---|---|
| `source_file`, `score_column`, `scale` | `benchmark_metadata.csv` | the file of a benchmark, the column that holds its score, and the factor that turns the score into a share of the maximum (0 to 1) |
| `Model version` | each benchmark file | the model name used by the Hub |
| `Source` | files whose name ends in `_external.csv` | who reported the score. Epoch's own runs have no `_external` in the file name and no `Source` |

### 2.2 Structure of the outputs in `processed/`

| File | One row is | Columns |
|---|---|---|
| `hub_scores_by_model.csv` | one score for a model, a benchmark and a kind of source | `model`, `release_year`, `access_group`, `benchmark`, `kind`, `source`, `score_pct` |
| `hub_model_coverage.csv` | one release year and access group | `release_year`, `access_group`, `models_in_capability_file`, `models_with_hub_scores` |
| `hub_name_matches.csv` | one Hub model version matched to one capability model | `hub_model_version`, `capability_model` |
| `hub_data_currency.csv` | this run | the dataset, the extraction time, when and how the ZIP was obtained, its sha256, the counts and a note |

`kind` is `epoch`, `third_party` or `developer`. `access_group` is `accessible` or `restricted`.


## 3. Set up the paths and settings

In [1]:
# Settings
from pathlib import Path
import hashlib, io, json, re, shutil, time, zipfile
from datetime import datetime, timezone
import pandas as pd
import requests

T0 = time.time()
FOLDER = "benchmark_hub"
cwd = Path.cwd()
if cwd.name == FOLDER:
    HERE = cwd
elif (cwd / FOLDER).exists():
    HERE = cwd / FOLDER
else:
    raise FileNotFoundError(f"Open this notebook from the folder {FOLDER}. Current folder: {cwd}")

RAW = HERE / "raw"                     # the ZIP as received
PROCESSED = HERE / "processed"         # tables this notebook writes
MANUAL = HERE / "manual"               # hand-made label table, and a hand-downloaded ZIP (optional)
STORY_DATA = HERE.parents[1] / "data"  # the story's own data folder. The notebook never writes here.
CAPABILITY = HERE.parent / "eci_scores" / "processed" / "model_capability.csv"
if not CAPABILITY.exists():
    CAPABILITY = STORY_DATA / "model_capability.csv"
    print("processed/model_capability.csv of eci_scores not found. Using", CAPABILITY)

HEADERS = {"User-Agent": "A3-student-project (academic use, no contact data stored)"}
TIMEOUT = 120
ZIP_URL = "https://epoch.ai/data/benchmark_data.zip"
ZIP_FILE = RAW / "benchmark_data.zip"
REFRESH = True         # False: reuse the ZIP already in raw/ (same snapshot as the page). True: try a fresh download first.
STRICT_LABELS = True   # True = stop if a source name is missing from manual/source_evidence_labels.csv
US = "United States of America"

for folder in (RAW, PROCESSED, MANUAL):
    folder.mkdir(exist_ok=True)
print("Notebook folder:", HERE)
print("Story data     :", STORY_DATA, "(copy files here by hand, see the last section)")


Notebook folder: d:\UTS MDSI\36104 (2026)\Assignment 3\A3P2\stories\2_gap_to_us_models\notebooks\benchmark_hub
Story data     : d:\UTS MDSI\36104 (2026)\Assignment 3\A3P2\stories\2_gap_to_us_models\data (copy files here by hand, see the last section)


## 4. Acquire the file

In [2]:
# Get the ZIP. Order: fresh download, earlier download in raw/, manual copy. Record how and when.
log_path = RAW / "retrieval_log.json"
log = json.loads(log_path.read_text(encoding="utf-8")) if log_path.exists() else {}
now = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
how = None

if REFRESH or not ZIP_FILE.exists():
    try:
        response = requests.get(ZIP_URL, headers=HEADERS, timeout=TIMEOUT)
        response.raise_for_status()
        ZIP_FILE.write_bytes(response.content)
        how, retrieved = "downloaded", now
    except Exception as err:
        print(f"Download of benchmark_data failed: {err}")

if how is None:
    manual_zip = MANUAL / ZIP_FILE.name
    candidates = [p for p in (ZIP_FILE, manual_zip) if p.exists()]
    if not candidates:
        raise FileNotFoundError(
            f"Could not get benchmark_data. Open {ZIP_URL} in a browser, save the file as '{ZIP_FILE.name}' "
            f"inside {MANUAL}, then run this cell again.")
    best = max(candidates, key=lambda p: p.stat().st_mtime)   # the newest copy wins
    if best == ZIP_FILE:
        how, retrieved = "reused", log.get("benchmark_data", {}).get("retrieved_utc", "not recorded")
        print(f"Using the earlier download {ZIP_FILE.name}")
    else:
        shutil.copy2(best, ZIP_FILE)
        how, retrieved = f"copied from manual folder {MANUAL.name}", now

digest = hashlib.sha256(ZIP_FILE.read_bytes()).hexdigest()
log["benchmark_data"] = {
    "url": ZIP_URL, "file": ZIP_FILE.name, "how": how, "retrieved_utc": retrieved,
    "bytes": ZIP_FILE.stat().st_size, "sha256": digest,
}
log_path.write_text(json.dumps(log, indent=2), encoding="utf-8")
print(json.dumps(log["benchmark_data"], indent=2))


{
  "url": "https://epoch.ai/data/benchmark_data.zip",
  "file": "benchmark_data.zip",
  "how": "downloaded",
  "retrieved_utc": "2026-11-20T13:06:42Z",
  "bytes": 2321522,
  "sha256": "d1bdd2ca3c77dee0e6369554e9504f3144fee207d1bcb6956cdc274b58b7daf2"
}


## 5. Read the benchmark list and every benchmark file inside the ZIP

In [3]:
# Read the benchmark list and every benchmark file inside the ZIP
archive = zipfile.ZipFile(ZIP_FILE)
names = set(archive.namelist())
metadata = pd.read_csv(archive.open("benchmark_metadata.csv"))
metadata = metadata.dropna(subset=["source_file", "score_column"]).reset_index(drop=True)
print(len(metadata), "benchmarks have a file and a score column")

frames, problems = [], []
for item in metadata.itertuples():
    if item.source_file not in names:
        problems.append((item.benchmark, "file missing in ZIP"))
        continue
    table = pd.read_csv(archive.open(item.source_file))
    if item.score_column not in table.columns:
        problems.append((item.benchmark, f"score column '{item.score_column}' missing"))
        continue
    score = pd.to_numeric(table[item.score_column], errors="coerce") * float(item.scale)
    is_external = item.source_file.endswith("_external.csv")   # Epoch's own runs have no "_external" in the file name
    frames.append(pd.DataFrame({
        "benchmark": item.benchmark,
        "hub_model_version": table["Model version"],
        "score": score,                       # share of the maximum, 0 to 1 (Hub scale)
        "is_external": is_external,
        "source_raw": table["Source"] if (is_external and "Source" in table.columns) else None,
    }))
rows = pd.concat(frames, ignore_index=True)
print("problems:", problems)
print(len(rows), "rows,", rows.benchmark.nunique(), "benchmarks,", int(rows.score.isna().sum()), "rows without a score,",
      int(rows.hub_model_version.isna().sum()), "rows without a model name")

58 benchmarks have a file and a score column
problems: []
5669 rows, 58 benchmarks, 8 rows without a score, 258 rows without a model name


## 6. Label every row by who produced the score

In [4]:
# Label every row by who produced the score
def clean_source(text):
    text = re.sub(r"\s+", " ", str(text)).strip()
    half = len(text) // 2
    return text[:half] if len(text) % 2 == 0 and text[:half] == text[half:] else text   # 'XX' typed twice -> 'X'

labels = pd.read_csv(MANUAL / "source_evidence_labels.csv")
label_of = dict(zip(labels.source_key, labels.evidence_type))
display_of = dict(zip(labels.source_key, labels.source))

def classify(row):
    if not row.is_external:
        return "epoch", "Epoch run"
    if pd.isna(row.source_raw):
        return "third_party", f"Source not stated ({row.benchmark})"
    key = clean_source(row.source_raw).casefold()
    if key not in label_of:
        return "unreviewed", clean_source(row.source_raw)
    return ("developer" if label_of[key] == "developer_report" else "third_party"), display_of[key]

kind_source = rows.apply(classify, axis=1, result_type="expand")
rows["kind"], rows["source"] = kind_source[0], kind_source[1]
missing = sorted(rows.loc[rows.kind == "unreviewed", "source"].unique())
print("sources not in the label table:", missing)
if STRICT_LABELS and missing:
    raise SystemExit("Add these sources to manual/source_evidence_labels.csv (developer_report or third_party), then run again.")
print(rows.kind.value_counts().to_string())

sources not in the label table: []
kind
third_party    2714
epoch          1704
developer      1251


## 7. Match Hub model names to the models in the capability file

In [5]:
# Match Hub model names to the models in the capability file
capability = pd.read_csv(CAPABILITY)
EFFORT = r"_(high|low|max|medium|xhigh|minimal|none|unknown|auto|default|thinking)$"
STYLE = r"-(instruct|chat)$"
DATE = r"-(\d{4}-\d{2}-\d{2}|\d{8})$"

def match_key(name):
    text = str(name).lower()
    text = re.sub(EFFORT, "", text)
    text = re.sub(STYLE, "", text)
    text = re.sub(DATE, "", text)
    return re.sub(r"[^a-z0-9]", "", text)

by_key = {}
for model in capability.model:
    by_key.setdefault(match_key(model), []).append(model)

versions = rows.hub_model_version.dropna().unique()
match = {v: by_key[match_key(v)][0] for v in versions if len(by_key.get(match_key(v), [])) == 1}
rows["model"] = rows.hub_model_version.map(match)
pd.DataFrame({"hub_model_version": list(match), "capability_model": list(match.values())}) \
  .sort_values(["capability_model", "hub_model_version"]).to_csv(PROCESSED / "hub_name_matches.csv", index=False)
ambiguous = [v for v in versions if len(by_key.get(match_key(v), [])) > 1]
print(len(match), "of", len(versions), "Hub versions matched;", len(ambiguous), "ambiguous (left unmatched)")
print(rows.model.nunique(), "of", len(capability), "capability models have at least one Hub row")

493 of 957 Hub versions matched; 0 ambiguous (left unmatched)
191 of 277 capability models have at least one Hub row


## 8. Access group and release year

In [6]:
# Access group and release year
open_weights = capability.epoch_accessibility_group.eq("Open weights")
group = pd.Series("excluded", index=capability.index)
group[capability.country.eq(US) & ~open_weights] = "restricted"
group[open_weights] = "accessible"
group[~open_weights & capability.country.notna() & ~capability.country.eq(US)] = "accessible"
capability["access_group"] = group
capability["release_year"] = capability.release_date.astype(str).str[:4]
print(capability.access_group.value_counts().to_string())

access_group
accessible    144
restricted    116
excluded       17


## 9. One score per model, benchmark and kind of source (highest kept), and save the tables

In [7]:
# One score per model, benchmark and kind of source (highest kept), and save the tables
scored = rows.dropna(subset=["model", "score"])
scored = scored.sort_values("score", ascending=False).drop_duplicates(["model", "benchmark", "kind"])
scored = scored.merge(capability[["model", "release_year", "access_group"]], on="model")
scored = scored[scored.access_group.isin(["accessible", "restricted"])]
out = scored[["model", "release_year", "access_group", "benchmark", "kind", "source", "score"]].copy()
out["score_pct"] = (out.score * 100).round(2)
out = out.drop(columns="score").sort_values(["release_year", "access_group", "model", "kind", "benchmark"])
out.to_csv(PROCESSED / "hub_scores_by_model.csv", index=False)

coverage = capability.groupby(["release_year", "access_group"]).model.nunique().rename("models_in_capability_file").reset_index()
with_scores = out.groupby(["release_year", "access_group"]).model.nunique().rename("models_with_hub_scores").reset_index()
coverage = coverage.merge(with_scores, how="left").fillna({"models_with_hub_scores": 0})
coverage.to_csv(PROCESSED / "hub_model_coverage.csv", index=False)
print(len(out), "scores,", out.model.nunique(), "models,", out.benchmark.nunique(), "benchmarks")
print(out.groupby(["release_year", "access_group", "kind"]).size().unstack(fill_value=0).to_string())
print(coverage.to_string(index=False))

2095 scores, 174 models, 57 benchmarks
kind                       developer  epoch  third_party
release_year access_group                               
2023         accessible          169      5           17
             restricted            8      2            3
2024         accessible           71     31           49
             restricted           19     41           67
2025         accessible            0     33           63
             restricted            4    144          243
2026         accessible            0    183          203
             restricted            1    331          408
release_year access_group  models_in_capability_file  models_with_hub_scores
        2023   accessible                         31                    23.0
        2023     excluded                         11                     0.0
        2023   restricted                          9                     2.0
        2024   accessible                         37                    21.0
      

## 10. Validate

In [8]:
# Validate
assert out.duplicated(["model", "benchmark", "kind"]).sum() == 0, "a model has two scores for one benchmark and kind"
assert out.groupby("model").release_year.nunique().max() == 1, "a model sits in two years"
assert out.score_pct.between(0, 100).all(), "a score is outside 0 to 100"
assert set(out.kind) <= {"epoch", "third_party", "developer"}
print("All checks passed")

All checks passed


## How to update, and what to do if the download is blocked

**Automatic.** `REFRESH = True` in Section 3 is already set. Run all cells. Section 4 downloads the ZIP and replaces `raw/benchmark_data.zip`. Run the eci_scores notebook of this story first, because Section 7 matches the Hub names against its `processed/model_capability.csv`.

**Manual download (use this if the run stops with `Download of benchmark_data failed`).**

1. Open https://epoch.ai/data/benchmark_data.zip in a browser.
2. Save the file as `benchmark_data.zip` inside the `manual/` folder (or in `raw/`). Keep the name.
3. Run the notebook again. The newest copy of the two is used.

**New sources.** If Section 6 stops with `Add these sources to manual/source_evidence_labels.csv`, add each source with the label `developer_report` or `third_party` and run again. The labels are a hand-made layer.

**Update check.** Open `raw/retrieval_log.json` and `processed/hub_data_currency.csv`. The page text that quotes counts of scores, models and benchmarks needs a check after a new download.


## Data currency: how up to date is this data?

In [9]:
try:
    cap_row = pd.read_csv(CAPABILITY.parent / "data_currency.csv").iloc[0]
    cap_note = (f"Release year and access group come from model_capability.csv (Epoch capability file, data to "
                f"{cap_row['data_covers_to']}, table made {cap_row['extracted_utc']}).")
except Exception:
    cap_note = "Release year and access group come from model_capability.csv. The date of that file is in its own data_currency.csv."
currency = pd.DataFrame([{
    "dataset": "Epoch Benchmarking Hub benchmark_data.zip (hub_scores_by_model.csv)",
    "extracted_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "zip_retrieved_utc": log["benchmark_data"]["retrieved_utc"], "zip_how": log["benchmark_data"]["how"],
    "zip_sha256": log["benchmark_data"]["sha256"],
    "scores": len(out), "models": out.model.nunique(), "benchmarks": out.benchmark.nunique(),
    "note": cap_note,
}])
currency.to_csv(PROCESSED / "hub_data_currency.csv", index=False)
for key, value in currency.iloc[0].items():
    print(f"{key:20s} {value}")


dataset              Epoch Benchmarking Hub benchmark_data.zip (hub_scores_by_model.csv)
extracted_utc        2026-11-20T13:06:44Z
zip_retrieved_utc    2026-11-20T13:06:42Z
zip_how              downloaded
zip_sha256           d1bdd2ca3c77dee0e6369554e9504f3144fee207d1bcb6956cdc274b58b7daf2
scores               2095
models               174
benchmarks           57
note                 Release year and access group come from model_capability.csv (Epoch capability file, data to 2026-10-07, table made 2026-11-08T21:21:38Z).


In [10]:
elapsed = time.time() - T0
print(f"Elapsed time of this run: {elapsed:.1f} seconds")
print("Files written:")
for p in sorted(PROCESSED.glob("*")):
    print(f"  {p.relative_to(HERE)}  ({p.stat().st_size:,} bytes)")


Elapsed time of this run: 1.8 seconds
Files written:
  processed\hub_data_currency.csv  (430 bytes)
  processed\hub_model_coverage.csv  (317 bytes)
  processed\hub_name_matches.csv  (16,832 bytes)
  processed\hub_scores_by_model.csv  (177,046 bytes)


## 11. Copy the tables into the story data folder (by hand)

The notebook does not write into the story's `data/` folder. After a run, copy the files listed below from `processed/` into `../../data/` by hand. The page reads only those copies. The cell below only prints. It shows which files to copy, their number of rows and whether the copy already in `data/` has the same content. `hub_name_matches.csv` stays here as a record of the name matches.

In [11]:
TO_COPY = ["hub_scores_by_model.csv", "hub_model_coverage.csv", "hub_data_currency.csv"]   # the page reads these three. hub_name_matches.csv stays here
for name in TO_COPY:
    made = PROCESSED / name
    rows_made = len(pd.read_csv(made))
    kept = STORY_DATA / name
    if not kept.exists():
        state = "not in the story data folder yet. Copy it."
    elif pd.read_csv(kept, keep_default_na=False).equals(pd.read_csv(made, keep_default_na=False)):
        state = "the copy in the story data folder has the same content. Nothing to do."
    else:
        state = "the copy in the story data folder is DIFFERENT. Copy the new file over it."
    print(f"{name}: {rows_made} rows. From {made.relative_to(HERE)} to {kept}. Status: {state}")


hub_scores_by_model.csv: 2095 rows. From processed\hub_scores_by_model.csv to d:\UTS MDSI\36104 (2026)\Assignment 3\A3P2\stories\2_gap_to_us_models\data\hub_scores_by_model.csv. Status: the copy in the story data folder is DIFFERENT. Copy the new file over it.
hub_model_coverage.csv: 10 rows. From processed\hub_model_coverage.csv to d:\UTS MDSI\36104 (2026)\Assignment 3\A3P2\stories\2_gap_to_us_models\data\hub_model_coverage.csv. Status: the copy in the story data folder has the same content. Nothing to do.
hub_data_currency.csv: 1 rows. From processed\hub_data_currency.csv to d:\UTS MDSI\36104 (2026)\Assignment 3\A3P2\stories\2_gap_to_us_models\data\hub_data_currency.csv. Status: the copy in the story data folder is DIFFERENT. Copy the new file over it.


## 12. Run time and limits

**Time to extract.** One ZIP of about 2.3 MB (2,321,522 bytes on 2026-11-20) and small tables in memory. The last cells print the measured time.

**Limits and known risks.**

1. The source labels (`manual/source_evidence_labels.csv`) and the model name matches (`processed/hub_name_matches.csv`) were read by one reader only and have no second reviewer.
2. A Hub model name that fits more than one capability model is left unmatched, so its scores are not counted.
3. Only models in the access groups `accessible` and `restricted` are kept. The group `excluded` is dropped.
4. One score is one model, one benchmark and one kind of source. A benchmark with two kinds of source for the same model counts twice. Versions of a model are merged and the highest score is kept.
5. A new ZIP can change the counts of scores, models and benchmarks that the page text quotes.

## 13. References

1. Epoch AI, Benchmarking Hub. https://epoch.ai/benchmarks
2. Epoch AI, Benchmark data download. https://epoch.ai/data/benchmark_data.zip
3. Creative Commons, Attribution 4.0 International. https://creativecommons.org/licenses/by/4.0/
